# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Respuesta:**

Gymnasium cumple el papel de **entorno**: proporciona el estado inicial, recibe las acciones, devuelve el siguiente estado, la recompensa y si el episodio terminó. En este caso, `gym.make("FrozenLake-v1", ...)` crea el ambiente.

El **agente no está definido como una clase separada**. Está representado por la Q-table (`Q`) junto con las funciones que seleccionan acciones y actualizan sus valores, principalmente `epsilon_greedy_policy()` y `train_q_learning()`. La Q-table contiene lo que el agente ha aprendido sobre el ambiente.


## 2. Crear FrozenLake


In [2]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset()

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda $Q[s,a]$?

**Respuesta:**

1. Debe tener **16 filas**, porque FrozenLake 4x4 tiene 16 estados posibles.
2. Debe tener **4 columnas**, porque existen 4 acciones: Left, Down, Right y Up.
3. Cada celda $Q[s,a]$ representa el **valor esperado de realizar la acción $a$ estando en el estado $s$** y continuar siguiendo una política óptima después. Un valor mayor indica que esa acción es más conveniente según lo aprendido.

Por eso la forma de la tabla es `(16, 4)`.


In [3]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Respuesta:**

Los ceros **no significan que todas las acciones sean malas**; significan que el agente todavía no tiene información aprendida sobre su utilidad. La tabla comienza sin conocimiento.

Si varias acciones tienen exactamente el mismo valor máximo, hay un empate. En este notebook se rompe el empate **aleatoriamente** entre las mejores acciones mediante `np.random.choice(best_actions)`. Esto evita favorecer siempre la primera acción de la tabla.


## 3. Ejecutar una transición


In [4]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 1
reward     = 0
next_state = 4
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(0,\ a,\ r,\ s')
$$

En la ejecución, el estado inicial de FrozenLake es `0`; los valores concretos de `a`, `r` y `s'` dependen de la acción aleatoria seleccionada en esa ejecución.

**¿De cuál de esos cuatro elementos no disponíamos directamente en Value Iteration cuando hablábamos de experiencia real?**

De la **experiencia real de transición**, porque Value Iteration trabaja con el modelo del ambiente: probabilidades de transición y recompensas esperadas. En Q-Learning observamos directamente la transición $(s,a,r,s')$ al interactuar con el ambiente.


## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [5]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [6]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


error: XDG_RUNTIME_DIR is invalid or not set in the environment.


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [7]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 0.0


### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [8]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

**Respuesta:**

Lo que cambia internamente es la **Q-table**.

- `Q_initial` contiene únicamente ceros: el agente no tiene conocimiento útil.
- `Q_early` ya contiene algunos valores aprendidos después de 50 episodios, por lo que puede empezar a preferir acciones que han producido mejores resultados.
- `Q_trained` ha recibido muchas más actualizaciones y, en el ambiente determinístico, aprende valores que permiten identificar un camino hacia la meta.

Por tanto, el ambiente no se vuelve más fácil: **mejora la información almacenada por el agente**, y esa información modifica la política greedy que utiliza para escoger sus acciones.


### 💬 Actividad 4 — Leer una fila de Q

Seleccionemos el estado $s=0$, que corresponde a la posición inicial de FrozenLake.

**Estado seleccionado:** `0`

**Valores Q:**

- Left: $Q(0,0)$
- Down: $Q(0,1)$
- Right: $Q(0,2)$
- Up: $Q(0,3)$

Después del entrenamiento, se puede inspeccionar la fila con:

```python
print(Q_trained[0])
```

La acción seleccionada será:

$$
rg\max_a Q(0,a)
$$

En el mapa 4x4 de FrozenLake, desde el estado inicial la acción óptima es **Down (1)** o, dependiendo de los valores aprendidos y del empate numérico, la acción que tenga el mayor Q. En un entrenamiento correctamente convergido, `Down` debe ser la mejor opción desde el estado 0.

**Interpretación:**

La fila muestra cuánto valora el agente cada acción posible desde el estado 0. La política greedy escoge la acción con mayor valor porque es la que el agente estima que conduce a una mayor recompensa futura.


In [9]:
# Inspección de la fila elegida para la Actividad 4
print('Q_trained[0] =', Q_trained[0])
print('Acción greedy desde el estado 0 =', greedy_policy(Q_trained, 0))

Q_trained[0] = [0.73509189 0.77378094 0.77378094 0.73509189]
Acción greedy desde el estado 0 = 1


## 5. Evaluar la política aprendida


In [10]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=rg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

Durante la evaluación queremos medir **la calidad de la política aprendida**, no seguir aprendiendo ni introducir decisiones aleatorias. Si exploráramos con $\epsilon$-greedy, algunas acciones serían escogidas al azar y una mala recompensa podría deberse a la exploración y no a una mala política. Por eso usamos explotación pura: escogemos la acción con mayor valor Q.


## 6. Experimento: FrozenLake estocástico


In [11]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.282
Std reward : 0.450


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

**Discusión:**

- **Ambiente:** con `is_slippery=False`, la acción elegida determina de forma predecible la transición (siempre que sea válida). Con `is_slippery=True`, el movimiento tiene incertidumbre: el agente puede terminar desplazándose en una dirección distinta de la elegida. Por eso una misma acción desde el mismo estado puede producir distintos estados siguientes.

- **Algoritmo:** la ecuación de actualización de Q-Learning **no cambia**:

$$
Q(s,a) \leftarrow Q(s,a)+lpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)ight]
$$

Lo que cambia es la experiencia observada $(s,a,r,s')$. En el ambiente estocástico, el mismo par $(s,a)$ puede generar diferentes $s'$, por lo que Q-Learning necesita más experiencias para aproximar correctamente el valor esperado.


# Cierre de clase

**1. ¿Qué almacena $Q(s,a)$?**

> Almacena una estimación del **retorno acumulado esperado** de ejecutar la acción $a$ desde el estado $s$ y continuar posteriormente de manera óptima.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> Sale de consultar la fila correspondiente al siguiente estado $s'$ en la Q-table y tomar el mayor valor entre todas las acciones posibles. Representa la mejor recompensa futura estimada desde ese estado.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Para equilibrar **exploración y explotación**. Con probabilidad $\epsilon$ el agente prueba acciones al azar para descubrir nuevas posibilidades; con probabilidad $1-\epsilon$ aprovecha lo que ya considera mejor. Si solo explotara desde el principio, podría quedarse con una política subóptima por falta de información.

**4. ¿Por qué Q-Learning es model-free?**

> Porque no necesita conocer previamente las probabilidades de transición ni la función de recompensa completa del ambiente. Aprende los valores Q directamente a partir de las experiencias obtenidas al interactuar con el ambiente.
